In [1]:
import numpy as np

from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense, Activation, Dropout
from tensorflow.python.keras.losses import CategoricalCrossentropy
from tensorflow.python.keras.optimizers import adam_v2

In [2]:
config = {
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
    "loss": CategoricalCrossentropy(),
    "optimizer": adam_v2.Adam(learning_rate=0.001),
}

In [3]:
from keras.datasets import mnist
from keras.utils import to_categorical

(X_train, y_train),(X_test, y_test) = mnist.load_data()
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

image_size = X_train.shape[1]
input_size = image_size * image_size

X_train = np.reshape(X_train, [-1, input_size])
X_train = X_train.astype('float32') / 255
X_test = np.reshape(X_test, [-1, input_size])
X_test = X_test.astype('float32') / 255

np.save('../../data/MNIST/train_data.npy', X_train)
np.save('../../data/MNIST/train_labels.npy', y_train)
np.save('../../data/MNIST/test_data.npy', X_test)
np.save('../../data/MNIST/test_labels.npy', y_test)

11490434/11490434 [==============================] - 7s 1us/step


In [7]:
def get_train_data():
    return np.load('../../data/MNIST/train_data.npy'), np.load('../../data/MNIST/train_labels.npy')

def get_test_data():
    return np.load('../../data/MNIST/test_data.npy'), np.load('../../data/MNIST/test_labels.npy')

In [8]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation('softmax'))
    return model

def train_model(model, X_train, y_train, loss, optimizer, epochs, batch_size):
    model.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size)
    return model

In [9]:
X_train, y_train = get_train_data()

In [10]:
model = create_model()
model = train_model(model, X_train, y_train, config["loss"], config["optimizer"], config["epochs"], config["batch_size"])

2023-06-23 15:11:52.537282: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 188160000 exceeds 10% of free system memory.


Epoch 1/20
469/469 [==============================] - 2s 3ms/step - loss: 0.4288 - accuracy: 0.8679
Epoch 2/20
469/469 [==============================] - 1s 3ms/step - loss: 0.1974 - accuracy: 0.9412
Epoch 3/20
469/469 [==============================] - 1s 3ms/step - loss: 0.1533 - accuracy: 0.9544
Epoch 4/20
469/469 [==============================] - 1s 3ms/step - loss: 0.1285 - accuracy: 0.9610
Epoch 5/20
469/469 [==============================] - 1s 3ms/step - loss: 0.1174 - accuracy: 0.9646
Epoch 6/20
469/469 [==============================] - 1s 3ms/step - loss: 0.1025 - accuracy: 0.9688
Epoch 7/20
469/469 [==============================] - 1s 3ms/step - loss: 0.0954 - accuracy: 0.9707
Epoch 8/20
469/469 [==============================] - 1s 3ms/step - loss: 0.0870 - accuracy: 0.9727
Epoch 9/20
469/469 [==============================] - 1s 3ms/step - loss: 0.0820 - accuracy: 0.9746
Epoch 10/20
469/469 [==============================] - 1s 3ms/step - loss: 0.0784 - accuracy: 0.9754

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0679 - accuracy: 0.9816

Test accuracy: 98.2%
